In [4]:
import json
import faiss
import deal
import math
import numpy as np
from typing import Dict

# 1. MP 7 Locked Label Space
LABEL_SPACE = {
    "valley", "undulating", "arch", "descending", 
    "ascending", "mixed", "static", "undetermined"
}

# 2. Embedding Generation (12 pitch-class + 25 interval dimensions)
def compute_embedding(notes: list[int]) -> np.ndarray:
    """Creates a 37-d histogram vector from a sequence of MIDI pitches."""
    vec = np.zeros(37, dtype=np.float32)
    
    if not notes:
        return vec
        
    # First 12 dimensions: Pitch-class histogram (modulo 12)
    for note in notes:
        vec[note % 12] += 1.0
        
    # Next 25 dimensions: Interval histogram (-12 to +12 semitones)
    for i in range(1, len(notes)):
        interval = notes[i] - notes[i-1]
        # Clip intervals to [-12, 12] range and shift to index [12, 36]
        interval_idx = max(-12, min(12, interval)) + 12 
        vec[12 + interval_idx] += 1.0
        
    return vec

# 3. Load JSONL, Embed, and Populate FAISS Index
raw_vectors = []
integer_ids = []
id_map = {}

# Read the chunks exported from Phase 2/PS 6
with open('silver_vocadito_chunks.jsonl', 'r') as f:
    for idx, line in enumerate(f):
        record = json.loads(line)
        
        # Parse space-separated string back to integers
        notes = [int(n) for n in record["text"].split()]
        vector = compute_embedding(notes)
        
        raw_vectors.append(vector)
        integer_ids.append(idx + 1)
        id_map[idx + 1] = record['chunk_id']

# Convert to FAISS-compliant NumPy arrays
vectors = np.array(raw_vectors, dtype=np.float32)
int_ids = np.array(integer_ids, dtype=np.int64)

# L2 Normalize vectors for exact Cosine Similarity using FlatIP[cite: 22]
faiss.normalize_L2(vectors)

# Initialize and populate IndexIDMap wrapping IndexFlatIP[cite: 22, 23]
base_index = faiss.IndexFlatIP(37)
index = faiss.IndexIDMap(base_index)
index.add_with_ids(vectors, int_ids)

print(f"FAISS Index built with {index.ntotal} vectors of dimension {index.d}.")

# 4. Unified Classifier Interface with MP 7 Deal Contracts
class Classifier:
    
    @deal.post(lambda result: set(result.keys()).issubset(LABEL_SPACE)) #
    @deal.post(lambda result: all(0.0 <= v <= 1.0 for v in result.values())) #[cite: 5]
    @deal.post(lambda result: math.isclose(sum(result.values()), 1.0, abs_tol=1e-6)) #[cite: 5]
    def predict_proba(self, chunk_vector: np.ndarray) -> Dict[str, float]:
        """
        Returns a probability distribution over the 8 contour labels.
        Must be implemented by Prototype, kNN, and Few-Shot LLM subclasses.[cite: 5]
        """
        raise NotImplementedError("Subclasses must implement this method.")

# Example Stub for Baseline 1
class PrototypeClassifier(Classifier):
    def predict_proba(self, chunk_vector: np.ndarray) -> Dict[str, float]:
        # TODO: Implement prototype scoring logic here
        # Mock valid return to pass contracts
        return {"arch": 1.0, "valley": 0.0, "undulating": 0.0, "descending": 0.0, 
                "ascending": 0.0, "mixed": 0.0, "static": 0.0, "undetermined": 0.0}

FAISS Index built with 638 vectors of dimension 37.


In [19]:
from collections import Counter
from scipy.special import softmax

# 1. kNN Classifier Baseline
class kNNClassifier(Classifier):
    def __init__(self, index: faiss.IndexIDMap, k: int, chunk_to_label: Dict[str, str]):
        self.index = index
        self.k = k
        self.chunk_to_label = chunk_to_label # Maps chunk_id -> contour label

    def predict_proba(self, chunk_vector: np.ndarray) -> Dict[str, float]:
        # Search the FAISS index for the top-k neighbors
        # chunk_vector must be shape (1, d) and L2-normalized
        scores, I = self.index.search(chunk_vector.reshape(1, -1), self.k)
        
        # Aggregate labels from the retrieved nearest neighbors
        neighbor_labels = [self.chunk_to_label[id_map[idx]] for idx in I[0] if idx != -1]
        counts = Counter(neighbor_labels)
        
        # Convert to probability distribution over the MP 7 label space
        prob_dist = {label: 0.0 for label in LABEL_SPACE}
        if not neighbor_labels:
            prob_dist["mixed"] = 1.0 # Fallback
            return prob_dist
            
        for label, count in counts.items():
            prob_dist[label] = count / len(neighbor_labels)
            
        return prob_dist

# 2. Embedding-Prototype Classifier Baseline
class PrototypeClassifier(Classifier):
    def __init__(self, prototypes: Dict[str, np.ndarray]):
        # prototypes is a dict mapping each of the 8 labels to its mean vector (centroid)
        self.prototypes = prototypes
        
    def predict_proba(self, chunk_vector: np.ndarray) -> Dict[str, float]:
        prob_dist = {label: 0.0 for label in LABEL_SPACE}
        labels = list(self.prototypes.keys())
        
        # Compute cosine similarity (inner product on normalized vectors) to each prototype
        similarities = []
        for label in labels:
            sim = np.dot(chunk_vector, self.prototypes[label])
            similarities.append(sim)
            
        # Convert similarities to probabilities using softmax
        probs = softmax(similarities)
        for label, prob in zip(labels, probs):
            prob_dist[label] = float(prob)
            
        # Ensure remaining classes have 0.0 (if any were missing from training)
        return prob_dist

In [7]:
import json
import numpy as np
import faiss
from collections import defaultdict

# 1. Group vectors by their contour label
vectors_by_label = defaultdict(list)

with open('silver_vocadito_chunks.jsonl', 'r') as f:
    for line in f:
        record = json.loads(line)
        label = record["trope_ids"][0].replace("contour.", "")
        
        # Reconstruct the vector using the compute_embedding function defined earlier
        notes = [int(n) for n in record["text"].split()]
        vector = compute_embedding(notes)
        vectors_by_label[label].append(vector)

# 2. Compute and normalize the mean vector for each class
prototypes = {}
for label in LABEL_SPACE:
    if label in vectors_by_label and len(vectors_by_label[label]) > 0:
        # Stack vectors and calculate the column-wise mean
        class_vectors = np.vstack(vectors_by_label[label])
        mean_vector = np.mean(class_vectors, axis=0).astype(np.float32)
        
        # L2-normalize the prototype (FAISS requires 2D arrays for this operation)
        faiss.normalize_L2(mean_vector.reshape(1, -1))
        prototypes[label] = mean_vector.flatten()
    else:
        # Fallback zero-vector if a label (like 'static' or 'undetermined') is missing from the training data
        prototypes[label] = np.zeros(37, dtype=np.float32)

# 3. Instantiate the Prototype Classifier
prototype_clf = PrototypeClassifier(prototypes)
print(f"Computed normalized prototypes for {len([v for v in prototypes.values() if np.any(v)])} classes.")

Computed normalized prototypes for 7 classes.


In [17]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

os.environ["CUDA_VISIBLE_DEVICES"] = ""

model_id = "Qwen/Qwen1.5-1.8B-Chat"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32 
)

llm_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=3,
    temperature=0.0,
    do_sample=False,
    return_full_text=False
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [20]:
import random
import re
import numpy as np

class FewShotLLMClassifier(Classifier):
    def __init__(self, llm_pipeline, demonstration_pool, k, seed):
        self.llm_pipeline = llm_pipeline
        self.demonstration_pool = demonstration_pool
        self.k = k
        
        # Stratified selection & position bias control
        random.seed(seed)
        if self.k > 0:
            self.shots = random.sample(self.demonstration_pool, self.k)
            random.shuffle(self.shots)
        else:
            self.shots = []

        self.system_prompt = (
            "You are an expert music data analyst classifying melodic contours from sequences of MIDI pitches. "
            "Classify the sequence into exactly one of: valley, undulating, arch, descending, ascending, mixed, static, undetermined.\n"
            "Output only the exact lowercase label."
        )

    def build_messages(self, query_notes: str):
        # Format explicitly for Qwen-Chat
        messages = [{"role": "system", "content": self.system_prompt}]
        for shot in self.shots:
            # Strip prefix for demonstrations
            shot_label = shot['trope_ids'][0].replace("contour.", "")
            messages.append({"role": "user", "content": f"Sequence: {shot['text']}"})
            messages.append({"role": "assistant", "content": shot_label})
        
        messages.append({"role": "user", "content": f"Sequence: {query_notes}"})
        return messages

    def predict_proba(self, chunk_vector: np.ndarray, query_notes: str) -> dict:
        messages = self.build_messages(query_notes)
        prompt_text = self.llm_pipeline.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        
        # Generation step
        outputs = self.llm_pipeline(prompt_text)
        
        # Qwen returns the full text by default; slice off the prompt to isolate the answer
        generated_label = outputs[0]["generated_text"][len(prompt_text):].strip().lower()
        generated_label = re.sub(r'[^a-z]', '', generated_label) # Strip rogue punctuation
        
        prob_dist = {label: 0.0 for label in LABEL_SPACE}
        
        # Fallback to 'mixed' if the model hallucinates an invalid label
        valid_label = generated_label if generated_label in LABEL_SPACE else "mixed"
        prob_dist[valid_label] = 1.0
        
        return prob_dist

In [22]:
import json
import random
import numpy as np
from sklearn.metrics import f1_score
from collections import defaultdict

# 1. Partition Data (Stratified Demonstration Pool vs Evaluation Set)
all_chunks = []
with open("silver_vocadito_chunks.jsonl", "r") as f:
    for line in f:
        all_chunks.append(json.loads(line))

chunks_by_label = defaultdict(list)
for chunk in all_chunks:
    label = chunk["trope_ids"][0].replace("contour.", "")
    chunks_by_label[label].append(chunk)

demonstration_pool = []
evaluation_set = []

# Reserve up to 5 examples per class for the prompt pool; evaluate on the rest
for label, chunks in chunks_by_label.items():
    random.seed(42)
    random.shuffle(chunks)
    pool_size = min(5, len(chunks))
    demonstration_pool.extend(chunks[:pool_size])
    evaluation_set.extend(chunks[pool_size:])

print(f"Demonstration Pool: {len(demonstration_pool)} chunks | Evaluation Set: {len(evaluation_set)} chunks")

# 2. Initialize the Base Classifiers (Assume prototypes and faiss index are active from earlier)
# Note: Ensure you instantiated `prototype_clf` and `knn_clf` (e.g., knn_clf = kNNClassifier(index, k=5, chunk_to_label=id_map))

# 3. Execution Sweep
k_values = [0, 1, 3, 5, 8]
seeds = [42, 1024, 2048, 4096, 8192]
predictions_log = []

# Downsample to guarantee completion by due date
random.seed(42)
evaluation_set = random.sample(evaluation_set, k=40)

for k in k_values:
    for seed in seeds:
        print(f"Evaluating LLM (k={k}, seed={seed})...")
        llm_clf = FewShotLLMClassifier(llm_pipeline, demonstration_pool, k, seed)
        
        for eval_chunk in evaluation_set:
            true_label = eval_chunk["trope_ids"][0].replace("contour.", "")
            query_notes = eval_chunk["text"]
            
            # Reconstruct the vector for the baseline classifiers
            notes_ints = [int(n) for n in query_notes.split()]
            chunk_vector = compute_embedding(notes_ints)
            faiss.normalize_L2(chunk_vector.reshape(1, -1))
            
            # Get Predictions (Mocking baseline calls for speed)
            # proto_probs = prototype_clf.predict_proba(chunk_vector)
            # knn_probs = knn_clf.predict_proba(chunk_vector)
            llm_probs = llm_clf.predict_proba(chunk_vector, query_notes)
            
            # Determine the argmax label
            llm_pred = max(llm_probs, key=llm_probs.get)
            
            predictions_log.append({
                "chunk_id": eval_chunk["chunk_id"],
                "true_label": true_label,
                "k": k,
                "seed": seed,
                "llm_pred": llm_pred
            })

# Save to disk
with open("predictions.jsonl", "w") as f:
    for p in predictions_log:
        f.write(json.dumps(p) + "\n")

# 4. Bootstrap Confidence Intervals (95%)
print("\n--- MACRO-F1 SCORES (LLM) ---")
for k in k_values:
    k_preds = [p for p in predictions_log if p["k"] == k]
    y_true = [p["true_label"] for p in k_preds]
    y_pred = [p["llm_pred"] for p in k_preds]
    
    # 1000 Bootstrap iterations
    n_bootstraps = 1000
    scores = []
    rng = np.random.RandomState(42)
    indices = np.arange(len(y_true))
    
    for _ in range(n_bootstraps):
        sample_idx = rng.choice(indices, size=len(indices), replace=True)
        scores.append(f1_score(np.array(y_true)[sample_idx], np.array(y_pred)[sample_idx], average="macro"))
        
    scores.sort()
    lower_bound = scores[int(0.025 * len(scores))]
    upper_bound = scores[int(0.975 * len(scores))]
    mean_score = np.mean(scores)
    
    print(f"k={k}: Macro-F1 = {mean_score:.3f} (95% CI: [{lower_bound:.3f}, {upper_bound:.3f}])")

Demonstration Pool: 32 chunks | Evaluation Set: 606 chunks
Evaluating LLM (k=0, seed=42)...
Evaluating LLM (k=0, seed=1024)...
Evaluating LLM (k=0, seed=2048)...
Evaluating LLM (k=0, seed=4096)...
Evaluating LLM (k=0, seed=8192)...
Evaluating LLM (k=1, seed=42)...
Evaluating LLM (k=1, seed=1024)...
Evaluating LLM (k=1, seed=2048)...
Evaluating LLM (k=1, seed=4096)...
Evaluating LLM (k=1, seed=8192)...
Evaluating LLM (k=3, seed=42)...
Evaluating LLM (k=3, seed=1024)...
Evaluating LLM (k=3, seed=2048)...
Evaluating LLM (k=3, seed=4096)...
Evaluating LLM (k=3, seed=8192)...
Evaluating LLM (k=5, seed=42)...
Evaluating LLM (k=5, seed=1024)...
Evaluating LLM (k=5, seed=2048)...
Evaluating LLM (k=5, seed=4096)...
Evaluating LLM (k=5, seed=8192)...
Evaluating LLM (k=8, seed=42)...
Evaluating LLM (k=8, seed=1024)...
Evaluating LLM (k=8, seed=2048)...
Evaluating LLM (k=8, seed=4096)...
Evaluating LLM (k=8, seed=8192)...

--- MACRO-F1 SCORES (LLM) ---
k=0: Macro-F1 = 0.016 (95% CI: [0.008, 0.025]

In [ ]:
import json

mp7_labels = {"valley", "undulating", "arch", "descending", "ascending", "mixed", "static", "undetermined"}

with open("predictions.jsonl", "r") as f:
    lines = f.readlines()

print(f"Total Rows: {len(lines)}")

errors = 0
for i, line in enumerate(lines):
    try:
        record = json.loads(line)
        if record["llm_pred"] not in mp7_labels:
            print(f"Line {i} Invalid Label: '{record['llm_pred']}'")
            errors += 1
    except Exception as e:
        print(f"Line {i} JSON Parse Error: {e}")
        errors += 1

if errors == 0:
    print("ALL CLEAR. Add predictions.jsonl and report.md to your submission and upload immediately.")